# BACKFILL Officers Data

## Overview
Add Officers Data
- Get id list per batch and group from corp processing entries
- Create a range for making Call to Function to load officers under parties

In [ ]:
%pip install pandas requests
%pip install sqlalchemy>=2.0
%pip install psycopg2-binary
%pip install python-dotenv

# Load Configurations

In [ ]:
import os
from datetime import datetime
from typing import Optional

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError, OperationalError
from sqlalchemy.engine import Engine
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
print("Environment variables loaded successfully.")

## Database Configuration

Configure connections to:
- **colin_extract**: Target database for `corp_processing` table

In [ ]:
DATABASE_CONFIG = {
    'business': {
        'username': os.getenv("DATABASE_USERNAME"),
        'password': os.getenv("DATABASE_PASSWORD"),
        'host': os.getenv("DATABASE_HOST"),
        'port': os.getenv("DATABASE_PORT"),
        'name': os.getenv("DATABASE_NAME")
    }
}

# Build connection URIs
for db_key, db_config in DATABASE_CONFIG.items():
    # Validate config
    missing_keys = [k for k, v in db_config.items() if v is None]
    if missing_keys:
        print(f"{db_key.upper()}: Missing environment variables for: {missing_keys}")

    # Build PostgreSQL URI
    uri = f"postgresql://{db_config['username']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['name']}"
    DATABASE_CONFIG[db_key] = {'uri': uri}

    print("Database configurations built successfully.")

TARGET_SCHEMA = os.getenv("TARGET_SCHEMA")
MIG_BATCH_ID = os.getenv("MIG_BATCH_ID")
ENVIRONMENTS = os.getenv("ENVIRONMENTS")
print("Service URLs and credentials loaded successfully.")

## Get Identifier for Batch and Group

In [ ]:
engines = {}

for db_key, config in DATABASE_CONFIG.items():
    try:
        print(f"Creating engine for {db_key.upper()}...")
        engine = create_engine(config['uri'])

        # Test connection
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))

        engines[db_key] = engine
        print(f"✓ {db_key.upper()} database engine created and tested successfully.")

    except OperationalError as e:
        print(f"✗ {db_key.upper()} database connection failed: {e}")
        raise
    except SQLAlchemyError as e:
        print(f"✗ {db_key.upper()} database engine creation failed: {e}")
        raise
    except Exception as e:
        print(f"✗ {db_key.upper()} unexpected error: {e}")
        raise

print("="*50)
print("All database engines ready for use.")
print("="*50)

In [ ]:
IDENTIFIERS_RANGE_QUERY = """
SELECT id, corp_num
FROM colin_extract.corp_processing cp
WHERE processed_status = 'COMPLETED'
AND mig_batch_id = :mig_batch_id
AND environment = :environment
-- LIMIT 1
"""

def query_identifiers(engine: Engine, mig_batch_id: int, environment: str) -> pd.DataFrame:
    try:
        with engine.connect() as conn:
            result = conn.execute(text(IDENTIFIERS_RANGE_QUERY), {"mig_batch_id": mig_batch_id, "environment": environment})
            identifiers_df = pd.DataFrame(result.fetchall(), columns=result.keys())
        print(f"✓ Successfully queried identifiers. Total records: {len(identifiers_df)}")
        return identifiers_df
    except SQLAlchemyError as e:
        print(f"✗ Error querying identifiers: {e}")
        raise
    except Exception as e:
        print(f"✗ Unexpected error querying identifiers: {e}")
        raise

identifier = query_identifiers(engines['business'], MIG_BATCH_ID, ENVIRONMENTS)
print(f"Total identifiers retrieved: {len(identifier)}")

In [ ]:
OFFICERS_RANGE_FUNCTION = """
SELECT public.colin_tombstone_officers_range(:environment, :first_id, :last_id);
"""
first_id = identifier['id'].iloc[0].item()
last_id = identifier['id'].iloc[-1].item()
print(f"Loading Officers from ID  {first_id} TO {last_id}")
def update_officers(engine: Engine, first_id: int, last_id: int, environment: str) -> pd.DataFrame:
    try:
        with engine.connect() as conn:
            result = conn.execute(text(OFFICERS_RANGE_FUNCTION), { "environment": environment,  "first_id": first_id, "last_id": last_id})
            conn.commit()
            value = result.scalar()
            print(f"Officers Result: {value}")
            return value
    except SQLAlchemyError as e:
        print(f"✗ Error querying officers data: {e}")
        raise
    except Exception as e:
        print(f"✗ Unexpected error querying identifiers: {e}")
        raise

officers_load = update_officers(engines['business'], first_id, last_id, ENVIRONMENTS)
print(f"Total updated officers: {officers_load}")

